# Triple-Barrier Parameter Analysis

This notebook provides a workflow for analyzing and visualizing the choice of parameters for the Triple-Barrier labeling method.

The process is as follows:
1. **Load Execution Price Data**: Fetch the realistic bid/ask execution prices for a given market and date range.
2. **Calculate & Visualize Volatility**: Compute a rolling volatility measure from the historical data, which will be used to set dynamic barriers.
3. **Define Barriers & Generate Labels**: Use the calculated volatility to define the triple barriers and generate the resulting labels.
4. **Analyze Label Distribution**: Summarize and plot the distribution of outcomes (Win, Loss, Timeout) to understand the impact of the chosen parameters.

## Setup

Import libraries and define the core parameters for the analysis.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import timedelta
from qate.core.model import Market, ExchangeName, Symbol
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange
from triple_barrier.triple_barrier import (
    TripleBarrierConfig,
    ts_index,
    calculate_volatility_and_spread,
    compute_labels,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
%matplotlib inline

PARQUET_STORE_ROOT_DIR = sys_env.get_parquet_root_dir()
MARKET = Market(ExchangeName.GMO, Symbol.XRP_JPY)
EXECUTION_SIZE = "150"
TIMEZONE = 'Asia/Tokyo' # Define the local timezone for analysis

DATA_START_DATE = "20251217"
DATA_END_DATE = "20251219"
DATA_DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)

# ######
# Sensitive AREA
#
# 1. Affect timeout percentage
# When RETURN_SHIFT_SEC == TIMEOUT_SEC 1.0 -> 60~70~%, 2.0 -> 80~90~%
# Hard to explain ... maybe later ask AI to summarize.
_BARRIER_MULTIPLIER = 0.7
#
# 2. Affect the imbalance of Win/Loss, longer -> more balanced but less predict power
_RETURN_SHIFT_SEC = 30
# 3. See 1
_TIMEOUT_SEC = 30
# 4. The minimum ratio of volatility to spread required for trading to be structurally profitable
_MIN_VOL_SPREAD_RATIO = 1.2

# ######
# Less sensitive AREA
_VOLATILITY_LOOKBACK_SEC = 1800
_EMA_SPAN_SECONDS = 600
_VOL_RESAMPLE_INTERVAL = "1s" # Reduce computation, '1s' is fast enough
_LABEL_RESAMPLE_INTERVAL = "100ms" # Reduce computation

CONFIG = TripleBarrierConfig(
    barrier_multiplier=_BARRIER_MULTIPLIER,
    return_shift_sec=_RETURN_SHIFT_SEC,
    timeout_sec=_TIMEOUT_SEC,
    min_vol_spread_ratio=_MIN_VOL_SPREAD_RATIO,
    volatility_lookback_sec=_VOLATILITY_LOOKBACK_SEC,
    ema_span_sec=_EMA_SPAN_SECONDS,
    vol_resample_interval=_VOL_RESAMPLE_INTERVAL,
    label_resample_interval=_LABEL_RESAMPLE_INTERVAL,
)

print(f"Setup complete. Analysis range: {DATA_DT_RANGE.start_str} to {DATA_DT_RANGE.end_str}")

## 1. Load Execution Price Data

Load the execution price data which contains the realistic bid, ask, and mid prices for our chosen market and order size.

In [ ]:

read_range = DtRange(
    DATA_DT_RANGE.start_dt - timedelta(hours=8),
    DATA_DT_RANGE.end_dt + timedelta(hours=8),
)

print(f"Loading execution prices for {MARKET.id} (size: {EXECUTION_SIZE})...")
price_store = ParquetStore(PARQUET_STORE_ROOT_DIR).create_execution_price(
    MARKET, execution_size_str=EXECUTION_SIZE)

df = price_store.read(read_range)
df = ts_index(df)
display(df)

## 2. Calculate & Visualize Volatility

Here we implement **Methodology 1**: calculating dynamic barriers based on recent volatility. We'll compute the rolling standard deviation of 1-minute returns.

In [ ]:

df = calculate_volatility_and_spread(df, CONFIG)
display(df)

print("Profit-take and Stop-loss levels (in %):")
display((df[['pt_pct', 'sl_pct']] * 100).describe())

# # MEMO to check if the result is healthy
display(df[["volatility_ema", "spread_pct_ema"]].corr())  # expecting big value 0.6~
display(df[["volatility_ema", "spread_pct_ema"]].diff().corr()) # expecting small value ~0.03


In [ ]:
# Plot the mid-price and the calculated volatility
fig, ax1 = plt.subplots(figsize=(15, 7))

color = 'tab:blue'
ax1.set_xlabel('Time')
ax1.set_ylabel('Spread', color=color)
ax1.plot(df.index, df['spread_pct_ema'], color=color, label='Spread')
ax1.tick_params(axis='y', labelcolor=color)

# ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
color = 'tab:red'
# ax2.set_ylabel('Volatility', color=color)
ax1.plot(df.index, df['volatility_ema'], color=color, alpha=0.7, label='Volatility')
# ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'{MARKET.symbol.name} Spread and Volatility')
fig.tight_layout()
plt.show()

## 2.5 Pre-Labeling Filter: Structural Profitability Analysis

Before running the computationally expensive labeling process, we'll analyze the structural profitability of the market over the selected period. We use the 'Gambler's Ruin' heuristic `P(Loss) ≈ 0.5 + 0.5 * (s/v)` to estimate the probability of loss based on the current relationship between transaction costs (spread `s`) and volatility (`v`).

If the average theoretical loss rate is higher than our defined `MAX_THEORETICAL_LOSS_RATE`, it suggests the market conditions are structurally unfavorable for our strategy's parameters, and we should consider adjusting them before proceeding.

In [ ]:
# To make spread and volatility comparable, we need to express spread as a percentage of price
# s_pct = spread / mid_price
# v_pct = BARRIER_MULTIPLIER * volatility
# P(Loss) = 0.5 + 0.5 * (s_pct / v_pct)

# Use the smoothed EMA values for a more stable calculation
# Parameters for the new structural profitability filter

MAX_THEORETICAL_LOSS_RATE = 0.65 # Set a target max loss rate for filtering (e.g., 0.65 = 65%)

s_pct_ema = df['spread_pct_ema']
v_pct_ema = df['volatility_ema']

# Calculate the theoretical probability of loss for each data point
# We cap the ratio at a high number to avoid extreme values from near-zero volatility
s_div_v = (s_pct_ema / v_pct_ema).clip(upper=5.0)
df['theoretical_loss_p'] = 0.5 + 0.5 * s_div_v

avg_loss_p = df['theoretical_loss_p'].mean()
avg_spread_ema = s_pct_ema.mean()
avg_volatility_ema = v_pct_ema.mean()

print("--- Structural Profitability Analysis ---")
print(f"Average EMA Spread: {avg_spread_ema:.5f}")
print(f"Average EMA Volatility (absolute): {avg_volatility_ema:.5f}")
print(f"Average Theoretical Loss Rate: {avg_loss_p:.2%}")
print(f"Your Maximum Acceptable Rate: {MAX_THEORETICAL_LOSS_RATE:.2%}")

# Plotting spread vs volatility
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df['volatility_ema'], y=s_pct_ema, hue=df['theoretical_loss_p'], palette='viridis', size=df['theoretical_loss_p'], sizes=(20, 400), alpha=0.6)
plt.axhline(y=avg_spread_ema, color='r', linestyle='--', label=f'Avg Spread ({avg_spread_ema:.5f})')
plt.axvline(x=avg_volatility_ema, color='g', linestyle='--', label=f'Avg Volatility ({avg_volatility_ema:.5f})')
plt.title('EMA Spread vs. EMA Volatility (Colored by Theoretical Loss Probability)')
plt.xlabel('EMA Volatility')
plt.ylabel('EMA Spread')
plt.legend()
plt.grid(True)
plt.show()

if avg_loss_p > MAX_THEORETICAL_LOSS_RATE:
    print("\nSTOP: The average theoretical loss rate exceeds your threshold.")
    print("Recommendation: Adjust parameters (e.g., increase BARRIER_MULTIPLIER, or run on a different time period) and re-run the notebook from the 'Setup' cell.")
else:
    print("\nGO: The average theoretical loss rate is within your threshold. Proceeding to label generation.")
    
df[['mid', 'theoretical_loss_p']].describe()

## 3. Define Barriers & Generate Labels

Now we use the calculated `volatility` series to define our dynamic barriers and generate labels. For simplicity, we will only generate labels for a **long scenario**.

In [ ]:
print(f"Generating labels with {CONFIG.timeout_sec}s timeout, "
      f"resampled to {CONFIG.label_resample_interval}...")
print(f"Original data points: {len(df)}")
result = compute_labels(df, CONFIG)

labels_df = result.labels_df
ambiguous_count = result.ambiguous_count
print(f"Resampled data points: {len(labels_df)}")
print(f"Ambiguous cases (both barriers hit in same bar): {ambiguous_count} "
      f"({ambiguous_count/len(labels_df)*100:.2f}%)")

# Join labels back - reindex to original df's index with forward fill
df['label'] = labels_df['label'].reindex(df.index, method='ffill')
df['pnl'] = labels_df['pnl'].reindex(df.index, method='ffill')
print("Label generation complete.")
df.head()

## 3.1 Filter by Vol/Spread Ratio

Filter to keep only data points where volatility exceeds spread by the configured ratio.
This ensures we only analyze periods with favorable structural conditions for trading.
The filtering is done AFTER label generation so that labels are computed for all data points first.

In [ ]:
# Apply the vol/spread filter from TripleBarrierConfig
from triple_barrier.triple_barrier import filter_by_vol_spread_ratio

original_count = len(df)
df_filtered = filter_by_vol_spread_ratio(df, CONFIG)
filtered_count = len(df_filtered)

print(f"Vol/Spread Filter (ratio={CONFIG.min_vol_spread_ratio}):")
print(f"  Before: {original_count:,} rows")
print(f"  After: {filtered_count:,} rows")
print(f"  Removed: {original_count - filtered_count:,} rows ({(original_count - filtered_count)/original_count*100:.1f}%)")

# Update labels_df for downstream analysis
labels_df = df_filtered[['label', 'pnl']].copy()
print(f"\nLabel distribution after filtering:")
print(labels_df['label'].value_counts().sort_index())

## 4. Analyze Label Distribution

Finally, we analyze the outcomes. How many of our potential trades resulted in a win, a loss, or a timeout? This is critical for understanding if the barrier parameters are well-chosen. If 99% of labels are timeouts, the barriers might be too wide or the timeout too short.

In [ ]:
# Exclude the last TIMEOUT_SEC of data (edge effect - no future data to check barriers)
cutoff_time = labels_df.index.max() - pd.Timedelta(seconds=CONFIG.timeout_sec)
labels_valid = labels_df.loc[labels_df.index <= cutoff_time, 'label'].dropna()

label_counts = labels_valid.value_counts().sort_index()

# Create a mapping for human-readable labels
label_map = {
    1: f'Win (PT = {CONFIG.barrier_multiplier}x Vol)',
    -1: f'Loss (SL = {CONFIG.barrier_multiplier}x Vol)',
    0: f'Timeout ({CONFIG.timeout_sec}s)'
}
label_counts.rename(index=label_map, inplace=True)

print(f"Label Distribution Summary (excluding last {CONFIG.timeout_sec}s of data):")
print(f"Total samples analyzed: {len(labels_valid):,}")
summary_df = pd.DataFrame({
    'count': label_counts,
    'percentage': (label_counts / label_counts.sum() * 100).round(2)
})
print(summary_df)

# Plot the distribution
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=summary_df.index, y=summary_df['count'])
ax.set_title('Triple-Barrier Label Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('Outcome')

# Add percentage labels to the bars
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2.,
            height + 0.1,
            f"{summary_df['percentage'].iloc[i]}%",
            ha="center", va="bottom")

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 5. PnL Analysis with Realistic Execution

The `compute_labels` function now calculates estimated PnL alongside labels, accounting for bid-ask spread:
- **Entry**: Buy at ASK price  
- **Exit**: Sell at BID price when barrier hit or timeout

The `labels_df` contains both `label` and `pnl` columns.

In [ ]:
# PnL is now calculated directly in compute_labels
# labels_df contains both 'label' and 'pnl' columns

# Exclude edge effect (last timeout_sec) for valid analysis
cutoff_time = labels_df.index.max() - pd.Timedelta(seconds=CONFIG.timeout_sec)
pnl_df = labels_df.loc[labels_df.index <= cutoff_time].copy()

# Calculate pnl in basis points using average mid price as reference
avg_mid = df['mid'].mean()
pnl_df['pnl_bps'] = (pnl_df['pnl'] / avg_mid) * 10000

print(f"Total samples: {len(pnl_df):,}")
print(f"\n{'='*60}")
print("PnL Summary by Label Type")
print('='*60)

for label_val, label_name in [(1, 'Win (label=1)'), (-1, 'Loss (label=-1)'), (0, 'Timeout (label=0)')]:
    subset = pnl_df[pnl_df['label'] == label_val]
    if len(subset) == 0:
        continue
    avg_pnl_bps = subset['pnl_bps'].mean() if 'pnl_bps' in subset.columns else 0
    print(f"\n{label_name}: {len(subset):,} trades")
    print(f"  Avg PnL: {subset['pnl'].mean():.4f} ({avg_pnl_bps:.2f} bps)")
    print(f"  Total PnL: {subset['pnl'].sum():.4f}")
    print(f"  Win rate (PnL > 0): {(subset['pnl'] > 0).mean():.2%}")

# Overall summary using LabelingResult properties
print(f"\n{'='*60}")
print("Overall Summary")
print('='*60)
print(f"Total PnL: {result.total_pnl:.4f}")
print(f"Avg PnL per trade: {result.avg_pnl:.6f}")
print(f"Overall win rate (PnL > 0): {(pnl_df['pnl'] > 0).mean():.2%}")

# Show the labels_df with pnl
display(labels_df.head(20))

In [ ]:
# Visualize PnL distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Filter out rows with zero pnl (no trade executed)
pnl_valid = pnl_df[pnl_df['pnl'] != 0].copy()

# 1. PnL distribution by label
ax1 = axes[0, 0]
for label_val, color, name in [(1, 'green', 'Win (label=1)'), (-1, 'red', 'Loss (label=-1)'), (0, 'orange', 'Timeout (label=0)')]:
    subset = pnl_valid[pnl_valid['label'] == label_val]['pnl_bps']
    if len(subset) > 0:
        ax1.hist(subset, bins=50, alpha=0.6, color=color, label=f'{name} (n={len(subset):,})')
ax1.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax1.set_xlabel('PnL (bps)')
ax1.set_ylabel('Count')
ax1.set_title('PnL Distribution by Label')
ax1.legend()

# 2. PnL histogram overall
ax2 = axes[0, 1]
ax2.hist(pnl_valid['pnl_bps'], bins=50, alpha=0.7, color='blue', edgecolor='black')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.axvline(x=pnl_valid['pnl_bps'].mean(), color='red', linestyle='-', linewidth=2, label=f'Mean: {pnl_valid["pnl_bps"].mean():.2f} bps')
ax2.set_xlabel('PnL (bps)')
ax2.set_ylabel('Count')
ax2.set_title('Overall PnL Distribution')
ax2.legend()

# 3. Cumulative PnL over time
ax3 = axes[1, 0]
pnl_sorted = pnl_valid.sort_index()
pnl_sorted['cumulative_pnl'] = pnl_sorted['pnl'].cumsum()
ax3.plot(pnl_sorted.index, pnl_sorted['cumulative_pnl'], linewidth=1)
ax3.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax3.set_xlabel('Time')
ax3.set_ylabel('Cumulative PnL')
ax3.set_title('Cumulative PnL Over Time')
ax3.tick_params(axis='x', rotation=45)

# 4. Average PnL by label (bar chart)
ax4 = axes[1, 1]
label_names = {1: 'Win', -1: 'Loss', 0: 'Timeout'}
colors = {1: 'green', -1: 'red', 0: 'orange'}
avg_pnl_by_label = pnl_valid.groupby('label')['pnl_bps'].mean()
labels_present = [l for l in [1, -1, 0] if l in avg_pnl_by_label.index]
x_labels = [label_names[l] for l in labels_present]
values = [avg_pnl_by_label[l] for l in labels_present]
bar_colors = [colors[l] for l in labels_present]
bars = ax4.bar(x_labels, values, color=bar_colors)
ax4.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax4.set_xlabel('Label')
ax4.set_ylabel('Average PnL (bps)')
ax4.set_title('Average PnL by Label')

# Add value labels on bars
for bar, val in zip(bars, values):
    ypos = bar.get_height() + (0.5 if val >= 0 else -1.5)
    ax4.text(bar.get_x() + bar.get_width()/2, ypos, f'{val:.1f}', ha='center', va='bottom' if val >= 0 else 'top')

plt.tight_layout()
plt.show()

# Show detailed statistics
print("\nDetailed PnL Statistics by Label (in bps):")
display(pnl_valid.groupby('label')['pnl_bps'].describe().round(2))

# Memo of conversation with AI

### What affects the win/loss rate

Setup
- Resample interval of volatility calculation = Tvol
- Timeout = Tvol
- Target profit = P

Factors
- Volatility = V
- Spread = S

```
Loss rate = 0.5 + 0.5 * (S/V)
```

The "Gambler's Ruin" Analogy

Imagine a particle starting at position 0 on a number line. At each step, it moves +1 or -1 with a 50% chance (a "symmetric random walk"). The game stops if the particle hits one of two walls:
* An upper "win" wall at position +a.
* A lower "loss" wall at position -b.

The well-known solution for the probability of hitting the lower wall at -b before hitting the upper wall at +a is:

P(Loss) = a / (a + b)

Applying it to Our Trading Model

Now, let's map our trading variables to this model:
* The "particle" is the true underlying price of the asset.
* a is the distance the price needs to travel up to trigger a win. As we established, this is v + s (it has to cover the barrier distance v plus the spread s).
* b is the distance the price needs to travel down to trigger a loss. This is v - s.

Now, we just plug our financial model's distances into the classic Gambler's Ruin formula:

1. Start with the formula:
P(Loss) = a / (a + b)

2. Substitute our distances:
P(Loss) = (v + s) / ( (v + s) + (v - s) )

3. Simplify the denominator:
P(Loss) = (v + s) / (2v)

4. Split the fraction:
P(Loss) = v/(2v) + s/(2v)

5. And you get the final expression:
P(Loss) = 0.5 + 0.5 * (s/v)

### The process/purpose to execute the notebook

Summary: A Dynamic Labeling Strategy Based on Structural Profitability

The proposed methodology is to evolve from a simple, static labeling process to a dynamic, regime-aware filtering system. The core principle is to only generate labels for entry points that have a favorable
structural probability of success, based on real-time market conditions.

This is achieved through the following steps:

1. Modeling Market Conditions: We will continuously track two key metrics:
* Transaction Cost: Represented by an Exponential Moving Average (EMA) of the bid-ask spread (s).
* Market Volatility: Represented by an EMA of the price volatility (v), calculated over a time horizon appropriate for the strategy.

2. Dynamic Entry Filtering: At each moment, we will use the formula derived from the "Gambler's Ruin" problem—`P(Loss) ≈ 0.5 + 0.5 * (s/v)`—to calculate the theoretical probability of a loss for a potential
trade initiated under the current market conditions.

3. Applying a Quality Threshold: We will define a hyperparameter, `X`, which represents the maximum acceptable theoretical loss rate. A triple-barrier label will only be generated if the calculated P(Loss) is
less than X. This acts as a filter to discard trades initiated during unfavorable regimes (e.g., high-spread/low-volatility periods).

4. Time Horizon Alignment: The volatility measurement window must be carefully tuned to align with the desired trading frequency (e.g., shorter windows for scalping strategies) to ensure the s/v ratio is
meaningful.

In essence, you are proposing to build a meta-model that pre-qualifies trades. It uses a simple, powerful heuristic to decide when the game is even worth playing, ensuring that the final labels are generated only
for high-quality opportunities. This is a superb approach that should dramatically increase the robustness and value of the resulting labels.
